# 09 — Compare And Interpret xG Models

Side-by-side evaluation of the baseline, skill-aware, form-aware, and combined
logistic-regression xG models on held-out EPL shots.

This notebook performs **no training and no feature engineering**. It loads saved
test-set predictions from notebooks 04 and 08, recomputes all comparison metrics
through one code path, and interprets model coefficients from saved artifacts.

| Model | Features | Prediction column |
|-------|----------|-------------------|
| baseline    | 7 baseline              | `xg_pred` |
| skill_aware | baseline + 5 skill      | `xg_pred_skill_aware` |
| form_aware  | baseline + 8 form       | `xg_pred_form_aware` |
| combined    | baseline + 5 skill + 8 form | `xg_pred_combined` |

In [1]:
import sys
print(sys.executable)

C:\Users\User\AppData\Local\Programs\Python\Python311\python.exe


In [2]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
    roc_curve,
)
from IPython.display import display

## Step 0: Constants and model registry

In [3]:
DATA_DIR   = Path("../data")
MODELS_DIR = Path("../models")

ID_COLS = ["league", "matchId", "playerId", "teamId", "eventSec", "matchPeriod"]
TARGET  = "is_goal"

BASELINE_FEATURES = [
    "distance_to_goal",
    "shot_angle_rad",
    "is_penalty",
    "is_direct_free_kick",
    "is_left_foot",
    "is_right_foot",
    "is_header",
]
SKILL_COLS = [
    "career_shots_before_shot",
    "career_goals_before_shot",
    "career_non_goal_shots_before_shot",
    "career_conversion_rate_before_shot",
    "has_prior_shot_history",
]
FORM_COLS = [
    "goals_last_5_matches",
    "shots_last_5_matches",
    "conversion_rate_last_5_matches",
    "matches_in_form_window",
    "xg_sum_last_5_matches",
    "goals_minus_xg_last_5_matches",
    "shots_per_match_last_5_matches",
    "xg_per_shot_last_5_matches",
]

MODELS = [
    {
        "name":      "baseline",
        "pred_col":  "xg_pred",
        "features":  BASELINE_FEATURES,
        "pred_path": DATA_DIR / "wyscout_test_xg_baseline_predictions.parquet",
        "artifact":  MODELS_DIR / "wyscout_xg_baseline_logreg.joblib",
    },
    {
        "name":      "skill_aware",
        "pred_col":  "xg_pred_skill_aware",
        "features":  BASELINE_FEATURES + SKILL_COLS,
        "pred_path": DATA_DIR / "wyscout_test_xg_skill_aware_predictions.parquet",
        "artifact":  MODELS_DIR / "wyscout_xg_skill_aware_logreg.joblib",
    },
    {
        "name":      "form_aware",
        "pred_col":  "xg_pred_form_aware",
        "features":  BASELINE_FEATURES + FORM_COLS,
        "pred_path": DATA_DIR / "wyscout_test_xg_form_aware_predictions.parquet",
        "artifact":  MODELS_DIR / "wyscout_xg_form_aware_logreg.joblib",
    },
    {
        "name":      "combined",
        "pred_col":  "xg_pred_combined",
        "features":  BASELINE_FEATURES + SKILL_COLS + FORM_COLS,
        "pred_path": DATA_DIR / "wyscout_test_xg_combined_predictions.parquet",
        "artifact":  MODELS_DIR / "wyscout_xg_combined_logreg.joblib",
    },
]
print("Constants OK")

Constants OK


## Step 1: Path validation

In [4]:
for model_cfg in MODELS:
    assert model_cfg["pred_path"].exists(), \
        f"Missing prediction file: {model_cfg['pred_path']}"
    assert model_cfg["artifact"].exists(), \
        f"Missing model artifact: {model_cfg['artifact']}"

BASELINE_METRICS_PATH = DATA_DIR / "wyscout_xg_baseline_metrics.csv"
IMPROVED_METRICS_PATH = DATA_DIR / "wyscout_xg_improved_metrics.csv"

assert BASELINE_METRICS_PATH.exists(), f"Missing {BASELINE_METRICS_PATH}"
assert IMPROVED_METRICS_PATH.exists(), f"Missing {IMPROVED_METRICS_PATH}"

print("Paths OK")

Paths OK


## Step 2: Load predictions and validate

All four test prediction files should contain the same EPL shots. We verify row
counts, league membership, duplicate-freeness, schema contracts, and cross-file
alignment of `is_goal` and the baseline `xg_pred` audit column.

In [5]:
pred_dfs = {}
for model_cfg in MODELS:
    pred_dfs[model_cfg["name"]] = pd.read_parquet(model_cfg["pred_path"])

n_test = len(pred_dfs["baseline"])

for model_cfg in MODELS:
    name = model_cfg["name"]
    df   = pred_dfs[name]

    assert len(df) == n_test, \
        f"{name} has {len(df)} rows, expected {n_test}"
    assert len(df) > 0, f"{name} is empty"
    assert set(df["league"].unique()) == {"England"}, \
        f"Wrong league in {name}: {df['league'].unique()}"
    assert df.duplicated(subset=ID_COLS).sum() == 0, \
        f"Duplicate rows in {name}"
    assert TARGET in df.columns, f"{TARGET} missing from {name}"
    assert model_cfg["pred_col"] in df.columns, \
        f"{model_cfg['pred_col']} missing from {name}"
    if name != "baseline":
        assert "xg_pred" in df.columns, \
            f"audit xg_pred missing from {name}"

# Cross-file alignment: is_goal and xg_pred must match across all files
def check_col_match(df_a, df_b, col, name_a, name_b):
    a = df_a.set_index(ID_COLS)[col].sort_index()
    b = df_b.set_index(ID_COLS)[col].sort_index()
    assert a.equals(b), f"{col} differs between {name_a} and {name_b}"

baseline_df = pred_dfs["baseline"]
for name in ["skill_aware", "form_aware", "combined"]:
    check_col_match(baseline_df, pred_dfs[name], "is_goal", "baseline", name)
    check_col_match(baseline_df, pred_dfs[name], "xg_pred", "baseline", name)

print(f"All predictions loaded and validated — {n_test:,} EPL test shots")
for name, df in pred_dfs.items():
    print(f"  {name}: {df.shape}")

All predictions loaded and validated — 8,881 EPL test shots
  baseline: (8881, 36)
  skill_aware: (8881, 42)
  form_aware: (8881, 45)
  combined: (8881, 50)


## Step 3: Build merged comparison table

One row per EPL shot with all four prediction columns side-by-side. Explicit column
selection keeps the merged table slim (11 columns).

In [6]:
comparison_df = pred_dfs["baseline"][ID_COLS + [TARGET, "xg_pred"]].copy()

for model_cfg in MODELS[1:]:  # skip baseline — already in comparison_df
    name     = model_cfg["name"]
    pred_col = model_cfg["pred_col"]
    comparison_df = comparison_df.merge(
        pred_dfs[name][ID_COLS + [pred_col]],
        on=ID_COLS, how="left", validate="one_to_one",
    )

# Post-merge assertions
assert len(comparison_df) == n_test, \
    f"Expected {n_test} rows, got {len(comparison_df)}"

for pred_col in [m["pred_col"] for m in MODELS]:
    assert pd.api.types.is_numeric_dtype(comparison_df[pred_col]), \
        f"{pred_col} is not numeric"
    assert comparison_df[pred_col].between(0, 1).all(), \
        f"{pred_col} has values outside [0, 1]"
    assert comparison_df[pred_col].notna().all(), \
        f"{pred_col} has null values"

assert pd.api.types.is_integer_dtype(comparison_df[TARGET]), \
    f"{TARGET} is not integer dtype"
assert set(comparison_df[TARGET].unique()) <= {0, 1}, \
    f"{TARGET} has unexpected values: {comparison_df[TARGET].unique()}"

comparison_df.to_parquet(
    DATA_DIR / "wyscout_xg_model_comparison.parquet", index=False
)

print(f"comparison_df: {comparison_df.shape}")
display(comparison_df.head())

comparison_df: (8881, 11)


,league,matchId,playerId,teamId,eventSec,matchPeriod,is_goal,xg_pred,xg_pred_skill_aware,xg_pred_form_aware,xg_pred_combined
0,England,2499719,25413,1609,94.595788,1H,1,0.137206,0.122826,0.114252,0.122490
1,England,2499719,26150,1631,179.854785,1H,0,0.117335,0.104749,0.096514,0.103708
2,England,2499719,14763,1631,254.745027,1H,1,0.421288,0.392990,0.379242,0.397953
3,England,2499719,7868,1609,425.824035,1H,0,0.044315,0.039946,0.036255,0.039344
4,England,2499719,7868,1609,815.462015,1H,0,0.020479,0.018898,0.017068,0.015508


## Step 4: Compute comparison metrics

All four models scored through one code path on the same target vector.

In [7]:
def metric_summary(y_true, y_prob, model_name):
    return {
        "model_name":        model_name,
        "n_shots":           int(len(y_true)),
        "goal_rate":         float(np.mean(y_true)),
        "roc_auc":           float(roc_auc_score(y_true, y_prob)),
        "average_precision": float(average_precision_score(y_true, y_prob)),
        "log_loss":          float(log_loss(y_true, y_prob)),
        "brier_score":       float(brier_score_loss(y_true, y_prob)),
        "mean_pred_xg":      float(np.mean(y_prob)),
    }

metrics_rows = []
for model_cfg in MODELS:
    metrics_rows.append(metric_summary(
        comparison_df[TARGET],
        comparison_df[model_cfg["pred_col"]],
        model_cfg["name"],
    ))

metrics_df = pd.DataFrame(metrics_rows)
display(metrics_df)

,model_name,n_shots,goal_rate,roc_auc,average_precision,log_loss,brier_score,mean_pred_xg
0,baseline,8881,0.111249,0.785701,0.374739,0.290893,0.083839,0.114052
1,skill_aware,8881,0.111249,0.787940,0.377225,0.289943,0.083638,0.114230
2,form_aware,8881,0.111249,0.787639,0.378000,0.290057,0.083668,0.114171
3,combined,8881,0.111249,0.788651,0.377501,0.289753,0.083638,0.114250


## Step 5: Cross-check against nb04/nb08 saved metrics

Freshly computed metrics must match the metrics saved by notebooks 04 and 08. If any
upstream prediction file was regenerated or corrupted, this cell catches it immediately.

Note: nb04 uses `split == "test_epl"` while nb08 uses `split == "test"`.

In [8]:
baseline_saved = pd.read_csv(BASELINE_METRICS_PATH)
improved_saved = pd.read_csv(IMPROVED_METRICS_PATH)

CROSS_CHECK_COLS = [
    "roc_auc", "average_precision", "log_loss", "brier_score",
    "mean_pred_xg", "n_shots", "goal_rate",
]

for model_cfg in MODELS:
    model_name = model_cfg["name"]
    fresh_rows = metrics_df[metrics_df["model_name"] == model_name]
    assert len(fresh_rows) == 1, \
        f"Expected 1 fresh row for {model_name}, got {len(fresh_rows)}"
    fresh = fresh_rows.to_dict("records")[0]

    if model_name == "baseline":
        saved_rows = baseline_saved[baseline_saved["split"] == "test_epl"]
    else:
        saved_rows = improved_saved[
            (improved_saved["model_name"] == model_name)
            & (improved_saved["split"] == "test")
        ]
    assert len(saved_rows) == 1, \
        f"Expected 1 {model_name} test row, got {len(saved_rows)}"
    saved = saved_rows.to_dict("records")[0]

    for col in CROSS_CHECK_COLS:
        if col in saved:
            assert np.isclose(fresh[col], saved[col], rtol=1e-4), \
                f"{model_name} {col} mismatch: fresh {fresh[col]} vs saved {saved[col]}"

    print(f"  {model_name}: cross-check passed")

print("\nAll cross-checks passed")

  baseline: cross-check passed
  skill_aware: cross-check passed
  form_aware: cross-check passed
  combined: cross-check passed

All cross-checks passed


## Step 6: Delta view relative to baseline

All deltas use "positive = better" convention. Sorted by `log_loss` ascending
(primary criterion).

In [9]:
baseline_rows = metrics_df[metrics_df["model_name"] == "baseline"]
assert len(baseline_rows) == 1, "Expected exactly 1 baseline row"
baseline_row = baseline_rows.to_dict("records")[0]

metrics_df["auc_gain"]           = metrics_df["roc_auc"]  - baseline_row["roc_auc"]
metrics_df["ap_gain"]            = metrics_df["average_precision"] - baseline_row["average_precision"]
metrics_df["log_loss_reduction"] = baseline_row["log_loss"]    - metrics_df["log_loss"]
metrics_df["brier_reduction"]    = baseline_row["brier_score"] - metrics_df["brier_score"]

metrics_df = metrics_df.sort_values("log_loss", ascending=True).reset_index(drop=True)

display(metrics_df)

metrics_df.to_csv(
    DATA_DIR / "wyscout_xg_model_comparison_metrics.csv", index=False
)
with open(DATA_DIR / "wyscout_xg_model_comparison_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_df.to_dict(orient="records"), f, indent=2)

print("Saved comparison metrics (CSV + JSON)")

,model_name,n_shots,goal_rate,roc_auc,average_precision,log_loss,brier_score,mean_pred_xg,auc_gain,ap_gain,log_loss_reduction,brier_reduction
0,combined,8881,0.111249,0.788651,0.377501,0.289753,0.083638,0.114250,0.002950,0.002762,0.001140,0.000201
1,skill_aware,8881,0.111249,0.787940,0.377225,0.289943,0.083638,0.114230,0.002240,0.002486,0.000950,0.000201
2,form_aware,8881,0.111249,0.787639,0.378000,0.290057,0.083668,0.114171,0.001938,0.003261,0.000836,0.000171
3,baseline,8881,0.111249,0.785701,0.374739,0.290893,0.083839,0.114052,0.000000,0.000000,0.000000,0.000000


Saved comparison metrics (CSV + JSON)


## Step 7: Calibration analysis

Decile calibration tables for all four models, built from the merged `comparison_df`.

In [10]:
cal_tables = []
for model_cfg in MODELS:
    model_name = model_cfg["name"]
    pred_col   = model_cfg["pred_col"]

    temp = comparison_df[[TARGET, pred_col]].copy()
    temp["xg_bin"] = pd.qcut(temp[pred_col], q=10, duplicates="drop")
    cal = (
        temp
        .groupby("xg_bin", observed=False)
        .agg(
            n_shots=(TARGET, "count"),
            mean_pred_xg=(pred_col, "mean"),
            observed_goal_rate=(TARGET, "mean"),
        )
        .reset_index()
    )
    cal.insert(0, "model_name", model_name)
    cal_tables.append(cal)

    print(f"\n--- {model_name} calibration table ---")
    display(cal)

cal_combined = pd.concat(cal_tables, ignore_index=True)
cal_combined.to_csv(
    DATA_DIR / "wyscout_xg_model_comparison_calibration_table.csv", index=False
)
print("\nSaved combined calibration table")


--- baseline calibration table ---


,model_name,xg_bin,n_shots,mean_pred_xg,observed_goal_rate
0,baseline,"(-0.000789, 0.0218]",894,0.015933,0.022371
1,baseline,"(0.0218, 0.0314]",884,0.026709,0.020362
2,baseline,"(0.0314, 0.0433]",888,0.037031,0.031532
3,baseline,"(0.0433, 0.0583]",891,0.050571,0.031425
4,baseline,"(0.0583, 0.075]",885,0.066730,0.054237
5,baseline,"(0.075, 0.0927]",888,0.083620,0.096847
6,baseline,"(0.0927, 0.119]",893,0.105163,0.097424
7,baseline,"(0.119, 0.155]",886,0.135759,0.150113
8,baseline,"(0.155, 0.237]",885,0.188011,0.202260
9,baseline,"(0.237, 0.981]",887,0.432027,0.406990



--- skill_aware calibration table ---


,model_name,xg_bin,n_shots,mean_pred_xg,observed_goal_rate
0,skill_aware,"(-0.00078, 0.0212]",889,0.015565,0.022497
1,skill_aware,"(0.0212, 0.0309]",888,0.026198,0.024775
2,skill_aware,"(0.0309, 0.0427]",888,0.036520,0.019144
3,skill_aware,"(0.0427, 0.057]",888,0.049678,0.045045
4,skill_aware,"(0.057, 0.0738]",888,0.065496,0.049550
5,skill_aware,"(0.0738, 0.093]",888,0.082765,0.083333
6,skill_aware,"(0.093, 0.118]",888,0.105037,0.101351
7,skill_aware,"(0.118, 0.156]",888,0.135180,0.163288
8,skill_aware,"(0.156, 0.239]",888,0.190495,0.201577
9,skill_aware,"(0.239, 0.983]",888,0.435479,0.402027



--- form_aware calibration table ---


,model_name,xg_bin,n_shots,mean_pred_xg,observed_goal_rate
0,form_aware,"(-0.000797, 0.0214]",889,0.015441,0.025872
1,form_aware,"(0.0214, 0.031]",888,0.026274,0.021396
2,form_aware,"(0.031, 0.043]",888,0.036676,0.021396
3,form_aware,"(0.043, 0.0577]",888,0.050104,0.045045
4,form_aware,"(0.0577, 0.0746]",888,0.065986,0.048423
5,form_aware,"(0.0746, 0.093]",888,0.083362,0.086712
6,form_aware,"(0.093, 0.119]",888,0.104873,0.105856
7,form_aware,"(0.119, 0.156]",888,0.135377,0.155405
8,form_aware,"(0.156, 0.238]",888,0.189758,0.199324
9,form_aware,"(0.238, 0.986]",888,0.433972,0.403153



--- combined calibration table ---


,model_name,xg_bin,n_shots,mean_pred_xg,observed_goal_rate
0,combined,"(-0.000795, 0.0211]",889,0.015337,0.022497
1,combined,"(0.0211, 0.0307]",888,0.026092,0.023649
2,combined,"(0.0307, 0.0425]",888,0.036385,0.019144
3,combined,"(0.0425, 0.057]",888,0.049553,0.045045
4,combined,"(0.057, 0.074]",888,0.065288,0.047297
5,combined,"(0.074, 0.0926]",888,0.082916,0.091216
6,combined,"(0.0926, 0.118]",888,0.104749,0.104730
7,combined,"(0.118, 0.157]",888,0.135389,0.150901
8,combined,"(0.157, 0.24]",888,0.190926,0.204955
9,combined,"(0.24, 0.986]",888,0.435976,0.403153



Saved combined calibration table


## Step 8: Overlaid calibration plot

In [11]:
COLORS  = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
MARKERS = ["o", "s", "^", "D"]

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Perfect calibration")

for model_cfg, color, marker in zip(MODELS, COLORS, MARKERS):
    prob_true, prob_pred = calibration_curve(
        comparison_df[TARGET],
        comparison_df[model_cfg["pred_col"]],
        n_bins=10,
        strategy="quantile",
    )
    ax.plot(prob_pred, prob_true, marker=marker, color=color,
            label=model_cfg["name"])

ax.set_xlabel("Predicted xG")
ax.set_ylabel("Observed goal rate")
ax.set_title("Calibration Curves — Held-out EPL Shots")
ax.legend()
ax.grid(True, alpha=0.3)

fig.savefig(
    DATA_DIR / "wyscout_xg_model_calibration_comparison.png",
    dpi=150, bbox_inches="tight",
)
plt.close(fig)
print("Saved calibration plot")

Saved calibration plot


## Step 9: Overlaid ROC plot

In [12]:
metrics_by_name = {
    row["model_name"]: row
    for row in metrics_df.to_dict("records")
}

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Random")

for model_cfg, color in zip(MODELS, COLORS):
    model_name = model_cfg["name"]
    pred_col   = model_cfg["pred_col"]
    auc_val    = float(metrics_by_name[model_name]["roc_auc"])

    fpr, tpr, _ = roc_curve(comparison_df[TARGET], comparison_df[pred_col])
    ax.plot(fpr, tpr, color=color, label=f"{model_name} (AUC = {auc_val:.4f})")

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Held-out EPL Shots")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)

fig.savefig(
    DATA_DIR / "wyscout_xg_model_roc_comparison.png",
    dpi=150, bbox_inches="tight",
)
plt.close(fig)
print("Saved ROC plot")

Saved ROC plot


## Step 10: Model coefficients

Load all four saved pipelines and extract standardised coefficients. The
`feature_names_in_` guard ensures coefficient labels match the feature order
used during training.

In [13]:
def get_feature_group(feature_name):
    if feature_name in SKILL_COLS:
        return "skill"
    elif feature_name in FORM_COLS:
        return "form"
    else:
        return "baseline"

coef_rows = []
for model_cfg in MODELS:
    model_name = model_cfg["name"]
    features   = model_cfg["features"]
    pipeline   = joblib.load(model_cfg["artifact"])

    assert hasattr(pipeline, "feature_names_in_"), \
        f"Pipeline for {model_name} has no feature_names_in_"
    assert list(pipeline.feature_names_in_) == features, \
        f"Feature order mismatch for {model_name}: " \
        f"pipeline has {list(pipeline.feature_names_in_)}"

    logreg = pipeline.named_steps["logreg"]
    coefs  = logreg.coef_[0]

    for feat, coef in zip(features, coefs):
        coef_rows.append({
            "model_name":      model_name,
            "feature":         feat,
            "feature_group":   get_feature_group(feat),
            "coefficient":     float(coef),
            "odds_ratio":      float(np.exp(coef)),
            "abs_coefficient": float(abs(coef)),
        })

coef_df = pd.DataFrame(coef_rows)
coef_df.to_csv(DATA_DIR / "wyscout_xg_model_coefficients.csv", index=False)
print(f"Saved coefficient table — {len(coef_df)} rows")

# View 1: Full combined model coefficients
print("\n--- Combined model: all 20 features (sorted by |coefficient|) ---")
display(
    coef_df[coef_df["model_name"] == "combined"]
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

# View 2: Skill/form features across improved models only
print("\n--- Skill & form features across improved models ---")
display(
    coef_df[
        (coef_df["feature_group"].isin(["skill", "form"]))
        & (coef_df["model_name"] != "baseline")
    ]
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

Saved coefficient table — 54 rows

--- Combined model: all 20 features (sorted by |coefficient|) ---


,model_name,feature,feature_group,coefficient,odds_ratio,abs_coefficient
0,combined,distance_to_goal,baseline,-0.866958,0.420228,0.866958
1,combined,shot_angle_rad,baseline,0.430350,1.537796,0.430350
2,combined,is_header,baseline,-0.278330,0.757047,0.278330
3,combined,is_penalty,baseline,0.255028,1.290497,0.255028
4,combined,is_direct_free_kick,baseline,0.183560,1.201487,0.183560
5,combined,shots_per_match_last_5_matches,form,0.146653,1.157952,0.146653
6,combined,is_left_foot,baseline,0.104202,1.109825,0.104202
7,combined,is_right_foot,baseline,0.102040,1.107428,0.102040
8,combined,career_goals_before_shot,skill,0.090587,1.094817,0.090587
9,combined,career_conversion_rate_before_shot,skill,0.061883,1.063837,0.061883



--- Skill & form features across improved models ---


,model_name,feature,feature_group,coefficient,odds_ratio,abs_coefficient
0,combined,shots_per_match_last_5_matches,form,0.146653,1.157952,0.146653
1,skill_aware,career_goals_before_shot,skill,0.103621,1.109179,0.103621
2,form_aware,shots_per_match_last_5_matches,form,0.094533,1.099146,0.094533
3,combined,career_goals_before_shot,skill,0.090587,1.094817,0.090587
4,combined,career_conversion_rate_before_shot,skill,0.061883,1.063837,0.061883
5,combined,shots_last_5_matches,form,-0.058465,0.943211,0.058465
6,combined,conversion_rate_last_5_matches,form,-0.045133,0.955871,0.045133
7,combined,matches_in_form_window,form,0.043338,1.044291,0.043338
8,combined,has_prior_shot_history,skill,-0.040724,0.960094,0.040724
9,form_aware,goals_minus_xg_last_5_matches,form,0.035907,1.036559,0.035907


## Step 11: Findings

**Interpretation hierarchy:** `log_loss` (primary) > `brier_score` + calibration
(corroborating) > `roc_auc` + `average_precision` (secondary diagnostics). If metrics
disagree, the disagreement is reported explicitly.

### Which model has the best EPL `log_loss`?

On the held-out EPL test set, the **combined** model performed best on the primary
criterion, achieving the lowest `log_loss` (`0.289753`) compared with `0.290893`
for the baseline. This indicates that combining long-term skill and short-term form
produced the strongest overall probability estimates among the four models tested.

### Do `brier_score` and calibration curves support the same conclusion?

The probability-quality metrics broadly support the same conclusion. The **combined**
and **skill-aware** models tied for the best `brier_score` (`0.083638`), both
improving on the baseline (`0.083839`). Calibration differences were small overall,
and the enhanced models were at least as well calibrated as the baseline, so the
improvement in `log_loss` did not come at the cost of clearly distorted probabilities.

### Does AUC / average precision agree?

The ranking-based diagnostics tell a slightly more nuanced story. The **combined**
model had the best `roc_auc` (`0.788651` vs `0.785701` baseline), which supports the
overall conclusion that it was the strongest model. However, the **form-aware** model
achieved the highest `average_precision` (`0.378000`), slightly above the combined
model (`0.377501`). This means the metrics do not agree perfectly: the combined model
is the best overall probability model, while the form-aware model shows the strongest
precision-recall behaviour on this imbalanced EPL test set.

### Which added features are most influential?

The coefficient analysis suggests that both long-term skill and short-term form add
useful signal beyond the baseline geometric features. In the combined model, the most
influential added feature was **`shots_per_match_last_5_matches`** (`coefficient ~= 0.147`),
followed by **`career_goals_before_shot`** (`~= 0.091`) and
**`career_conversion_rate_before_shot`** (`~= 0.062`). This indicates that recent
shooting involvement and longer-term finishing history both contribute meaningfully to
shot-level prediction.

### Are gains meaningful?

The gains are **modest but consistent**. The absolute improvements over the baseline
are small, but they appear across the main probability-quality metrics: `log_loss`,
`brier_score`, and calibration remain at least as good as the baseline, while
`roc_auc` also improves. For a logistic-regression xG framework, these are meaningful
gains because they show that player-level skill and form features add predictive signal
beyond standard geometric shot inputs without sacrificing reliability.

### Overall conclusion

Overall, the results suggest that player-level information does improve expected goals
prediction beyond the baseline geometry model. The **combined** model provides the
strongest overall EPL performance, the **skill-aware** model is a very close second,
and the **form-aware** model adds useful short-term signal even though it does not
beat the combined model on the primary probability metrics.


## Step 12: Reload verification

Reload all saved outputs and verify schemas, row counts, dtypes, and value ranges.

In [14]:
print("Reload verification:")

# 1. Comparison parquet
comp_check = pd.read_parquet(DATA_DIR / "wyscout_xg_model_comparison.parquet")
assert len(comp_check) == n_test, f"Expected {n_test} rows, got {len(comp_check)}"
assert comp_check.shape[1] == 11, f"Expected 11 columns, got {comp_check.shape[1]}"
for pred_col in [m["pred_col"] for m in MODELS]:
    assert pd.api.types.is_numeric_dtype(comp_check[pred_col])
    assert comp_check[pred_col].between(0, 1).all()
assert pd.api.types.is_integer_dtype(comp_check[TARGET])
print("  comparison parquet: OK")

# 2. Metrics CSV
metrics_check = pd.read_csv(DATA_DIR / "wyscout_xg_model_comparison_metrics.csv")
assert len(metrics_check) == 4, f"Expected 4 rows, got {len(metrics_check)}"
expected_metric_cols = {
    "model_name", "n_shots", "goal_rate", "roc_auc", "average_precision",
    "log_loss", "brier_score", "mean_pred_xg",
    "auc_gain", "ap_gain", "log_loss_reduction", "brier_reduction",
}
assert expected_metric_cols.issubset(metrics_check.columns), \
    f"Missing columns: {expected_metric_cols - set(metrics_check.columns)}"
assert set(metrics_check["model_name"]) == {"baseline", "skill_aware", "form_aware", "combined"}
print("  metrics CSV: OK")

# 3. Metrics JSON
with open(DATA_DIR / "wyscout_xg_model_comparison_metrics.json", "r", encoding="utf-8") as f:
    metrics_json = json.load(f)
assert len(metrics_json) == 4
assert set(metrics_json[0].keys()) == set(metrics_check.columns)
print("  metrics JSON: OK")

# 4. Calibration table
cal_check = pd.read_csv(DATA_DIR / "wyscout_xg_model_comparison_calibration_table.csv")
assert set(cal_check["model_name"]) == {"baseline", "skill_aware", "form_aware", "combined"}
for name in ["baseline", "skill_aware", "form_aware", "combined"]:
    assert len(cal_check[cal_check["model_name"] == name]) >= 1
expected_cal_cols = {"model_name", "xg_bin", "n_shots", "mean_pred_xg", "observed_goal_rate"}
assert expected_cal_cols.issubset(cal_check.columns)
print("  calibration table: OK")

# 5. Coefficient table
coef_check = pd.read_csv(DATA_DIR / "wyscout_xg_model_coefficients.csv")
assert set(coef_check["model_name"]) == {"baseline", "skill_aware", "form_aware", "combined"}
assert set(coef_check["feature_group"]) == {"baseline", "skill", "form"}
expected_coef_rows = sum(len(m["features"]) for m in MODELS)  # 7 + 12 + 15 + 20 = 54
assert len(coef_check) == expected_coef_rows, \
    f"Expected {expected_coef_rows} coefficient rows, got {len(coef_check)}"
print("  coefficient table: OK")

# 6. PNG plots
for png_name in [
    "wyscout_xg_model_calibration_comparison.png",
    "wyscout_xg_model_roc_comparison.png",
]:
    png_path = DATA_DIR / png_name
    assert png_path.exists(), f"Missing {png_path}"
    assert png_path.stat().st_size > 0, f"Empty {png_path}"
print("  PNG plots: OK")

print("\nAll reload checks passed")

Reload verification:
  comparison parquet: OK
  metrics CSV: OK
  metrics JSON: OK
  calibration table: OK
  coefficient table: OK
  PNG plots: OK

All reload checks passed
